In [34]:
import numpy as np
import pandas as pd

In [35]:
# Create Figure 1. synthetic bottom data
ds = pd.date_range(start='2000-01-01', end='2000-08-01', freq='MS')
y_base = np.arange(1,9)
r1 = y_base * (10**1)
r2 = y_base * (10**1)
r3 = y_base * (10**2)
r4 = y_base * (10**2)

ys = np.concatenate([r1, r2, r3, r4])
ds = np.tile(ds, 4)
unique_ids = ['r1'] * 8 + ['r2'] * 8 + ['r3'] * 8 + ['r4'] * 8
top_level = 'Australia'
middle_level = ['State1'] * 16 + ['State2'] * 16
bottom_level = unique_ids

bottom_df = dict(ds=ds,
                 top_level=top_level, 
                 middle_level=middle_level, 
                 bottom_level=bottom_level,
                 y=ys)
bottom_df = pd.DataFrame(bottom_df)
bottom_df.groupby('bottom_level').head(2)

,ds,top_level,middle_level,bottom_level,y
0,2000-01-01,Australia,State1,r1,10
1,2000-02-01,Australia,State1,r1,20
8,2000-01-01,Australia,State1,r2,10
9,2000-02-01,Australia,State1,r2,20
16,2000-01-01,Australia,State2,r3,100
17,2000-02-01,Australia,State2,r3,200
24,2000-01-01,Australia,State2,r4,100
25,2000-02-01,Australia,State2,r4,200


In [38]:
from hierarchicalforecast.utils import aggregate

In [39]:
# Create hierarchical structure and constraints
hierarchy_levels = [['top_level'],
                    ['top_level', 'middle_level'],
                    ['top_level', 'middle_level', 'bottom_level']]
Y_hier_df, S_df, tags = aggregate(df=bottom_df, spec=hierarchy_levels)
print('S_df.shape', S_df.shape)
print('Y_hier_df.shape', Y_hier_df.shape)
print("tags['top_level']", tags['top_level'])

S_df.shape (7, 5)
Y_hier_df.shape (56, 3)
tags['top_level'] ['Australia']


In [44]:
from statsforecast.models import Naive
from statsforecast.core import StatsForecast

In [45]:
# Split train/test sets
Y_test_df  = Y_hier_df.groupby('unique_id', as_index=False).tail(4)
Y_train_df = Y_hier_df.drop(Y_test_df.index)

# Compute base Naive predictions
# Careful identifying correct data freq, this data monthly 'M'
fcst = StatsForecast(models=[Naive()],
                     freq='MS', n_jobs=-1)
Y_hat_df = fcst.forecast(df=Y_train_df, h=4, fitted=True)
Y_fitted_df = fcst.forecast_fitted_values()

In [46]:
from hierarchicalforecast.methods import BottomUp
from hierarchicalforecast.core import HierarchicalReconciliation

In [48]:
Y_hat_df

,unique_id,ds,Naive
0,Australia,2000-05-01,880.0
1,Australia,2000-06-01,880.0
2,Australia,2000-07-01,880.0
3,Australia,2000-08-01,880.0
4,Australia/State1,2000-05-01,80.0
5,Australia/State1,2000-06-01,80.0
6,Australia/State1,2000-07-01,80.0
7,Australia/State1,2000-08-01,80.0
8,Australia/State1/r1,2000-05-01,40.0
9,Australia/State1/r1,2000-06-01,40.0


In [50]:
Y_rec_df

,unique_id,ds,Naive,Naive/BottomUp
0,Australia,2000-05-01,880.0,880.0
1,Australia,2000-06-01,880.0,880.0
2,Australia,2000-07-01,880.0,880.0
3,Australia,2000-08-01,880.0,880.0
4,Australia/State1,2000-05-01,80.0,80.0
5,Australia/State1,2000-06-01,80.0,80.0
6,Australia/State1,2000-07-01,80.0,80.0
7,Australia/State1,2000-08-01,80.0,80.0
8,Australia/State2,2000-05-01,800.0,800.0
9,Australia/State2,2000-06-01,800.0,800.0


In [47]:
# You can select a reconciler from our collection
reconcilers = [BottomUp()] # MinTrace(method='mint_shrink')
hrec = HierarchicalReconciliation(reconcilers=reconcilers)

Y_rec_df = hrec.reconcile(Y_hat_df=Y_hat_df, 
                          Y_df=Y_fitted_df,
                          S_df=S_df, tags=tags)
Y_rec_df.groupby('unique_id').head(2)

,unique_id,ds,Naive,Naive/BottomUp
0,Australia,2000-05-01,880.0,880.0
1,Australia,2000-06-01,880.0,880.0
4,Australia/State1,2000-05-01,80.0,80.0
5,Australia/State1,2000-06-01,80.0,80.0
8,Australia/State2,2000-05-01,800.0,800.0
9,Australia/State2,2000-06-01,800.0,800.0
12,Australia/State1/r1,2000-05-01,40.0,40.0
13,Australia/State1/r1,2000-06-01,40.0,40.0
16,Australia/State1/r2,2000-05-01,40.0,40.0
17,Australia/State1/r2,2000-06-01,40.0,40.0
